In [1]:
import os
import sys
sys.path.append("..")
from helpers.fusion import FusionModule
from helpers.image_encoder import ImageEncoder
from helpers.text_encoder  import TextEncoder
from helpers.projection import ProjectionModule
import torch
import torch.nn as nn

In [2]:
img_dir = "../../task2/data/exp_data/cat_01.jpeg"

In [ ]:
import torch
model_for_image = torch.hub.load('facebookresearch/dino:main', 'dino_vits16')
model_for_image.eval()

Using cache found in /home/saad/.cache/torch/hub/facebookresearch_dino_main


VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (blocks): ModuleList(
    (0-11): 12 x Block(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (norm): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
  (head): Identity()
)

In [ ]:
image_encoder = ImageEncoder(model_for_image)
encoded_img = image_encoder.encode(img_dir)
print(encoded_img.shape)  # should be [1, 384]

torch.Size([1, 384])


In [5]:
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
model = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")

In [6]:

text_encoder = TextEncoder(model, tokenizer)
encoded_text = text_encoder.encode("Patient has mild pneumonia.")
print(encoded_text.shape)  # should be [1, hidden_size]

torch.Size([1, 768])


In [7]:
fusion_model = FusionModule()
fused_embedding = fusion_model.fuse(encoded_img, encoded_text)
print(fused_embedding.shape)  # should be [1, 1152]

torch.Size([1, 1152])


In [8]:
icd_embedding = text_encoder.encode("This is a case of acute bronchitis.")
print(icd_embedding.shape)  # should be [1, hidden_size]

torch.Size([1, 768])


In [9]:
len(icd_embedding[0])

768

In [10]:
# Project icd embedding and fused embedding to same space using mlp
# Simple projection
proj_dim = 512
mlp_icd = nn.Linear(len(icd_embedding[0]), proj_dim)
mlp_fused = nn.Linear(len(fused_embedding[0]), proj_dim)

icd_embedding_proj = mlp_icd(icd_embedding)  # shape: (1, dim2)
fused_embedding_proj = mlp_fused(fused_embedding)  # shape: (1, dim2)



In [11]:
print(icd_embedding_proj.shape)  # should be [1, proj_dim]
print(fused_embedding_proj.shape)  # should be [1, proj_dim]

torch.Size([1, 512])
torch.Size([1, 512])


In [12]:
proj_module_fused = ProjectionModule(input_dim=len(fused_embedding[0]), output_dim=512, method="linear")
fused_embedding_proj2 = proj_module_fused.project(fused_embedding)

proj_module_icd = ProjectionModule(input_dim=len(icd_embedding[0]), output_dim=512, method="linear")
icd_embedding_proj2 = proj_module_icd.project(icd_embedding)
print(fused_embedding_proj2.shape)  # should be [1, 512]
print(icd_embedding_proj2.shape)  # should be [1, 512]

torch.Size([1, 512])
torch.Size([1, 512])
